# GJR-GARCH Inverse Volatility Strategy with SMA50 Trend Filter
## Top 10 Cryptocurrencies Portfolio

---

**Strategy Description:**
- **Position Sizing**: Inverse volatility using GJR-GARCH conditional variance forecasts
- **Entry Filter**: Price must be above SMA(50) to take long positions
- **Universe**: Top 10 cryptocurrencies by market capitalization
- **Rebalancing**: Daily allocation adjustments based on volatility estimates

**Key Principles from Bootcamp:**
- Proper train/validation splits to avoid overfitting
- Lookahead bias prevention (signals shifted by 1 bar)
- Transaction cost modeling
- Robust performance metrics

---

In [ ]:
# INSTALLATION CELL - Run only if packages are not installed
# !pip install yfinance
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy
# !pip install arch  # For GJR-GARCH models
# !pip install matplotlib
# !pip install seaborn

In [ ]:
# IMPORTS
import yfinance as yf
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from arch import arch_model
from datetime import datetime, timedelta

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12

print("All libraries loaded successfully!")

## 1. DATA INGESTION - Top 10 Cryptocurrencies

We'll download historical data for the top 10 cryptocurrencies by market cap. 
The universe includes: BTC, ETH, BNB, XRP, ADA, SOL, DOGE, DOT, MATIC, AVAX

In [ ]:
# CONFIGURATION
# ============================================================================

# Top 10 Cryptocurrencies (by historical market cap)
CRYPTO_TICKERS = [
    'BTC-USD',   # Bitcoin
    'ETH-USD',   # Ethereum
    'BNB-USD',   # Binance Coin
    'XRP-USD',   # Ripple
    'ADA-USD',   # Cardano
    'SOL-USD',   # Solana
    'DOGE-USD',  # Dogecoin
    'DOT-USD',   # Polkadot
    'MATIC-USD', # Polygon
    'AVAX-USD',  # Avalanche
]

# Date range
START_DATE = '2020-01-01'  # Start from 2020 for sufficient data
END_DATE = None  # None = today

# Strategy parameters
SMA_PERIOD = 50           # Trend filter period
GARCH_LOOKBACK = 252      # Days for GARCH estimation (1 year)
VOL_LOOKBACK = 14         # Days for simple volatility fallback
REBALANCE_FREQ = 'D'      # Daily rebalancing
MAX_LEVERAGE = 2.0        # Maximum portfolio leverage
TARGET_VOL = 0.15         # Annual target volatility (15%)

# Backtesting parameters
INITIAL_CAPITAL = 100_000
FEES = 0.001              # 0.1% trading fees
SLIPPAGE = 0.0005         # 0.05% slippage

# Train/Validation split
TRAIN_RATIO = 0.60

print("Configuration loaded!")
print(f"Universe: {len(CRYPTO_TICKERS)} cryptocurrencies")
print(f"Start Date: {START_DATE}")
print(f"SMA Period: {SMA_PERIOD}")
print(f"Target Volatility: {TARGET_VOL:.1%}")

In [ ]:
# DOWNLOAD CRYPTOCURRENCY DATA
# ============================================================================

def download_crypto_data(tickers, start_date, end_date=None):
    """
    Download historical data for multiple cryptocurrencies from yfinance.
    Includes data validation and quality checks.
    """
    print(f"Downloading data for {len(tickers)} cryptocurrencies...")
    print("="*70)
    
    # Download all tickers at once for efficiency
    raw_data = yf.download(
        tickers, 
        start=start_date, 
        end=end_date,
        interval='1d',
        auto_adjust=True,
        progress=True
    )
    
    # Extract close prices
    if isinstance(raw_data.columns, pd.MultiIndex):
        close_prices = raw_data['Close']
        volume = raw_data['Volume']
        high = raw_data['High']
        low = raw_data['Low']
    else:
        close_prices = raw_data[['Close']]
        close_prices.columns = tickers
        volume = raw_data[['Volume']]
        high = raw_data[['High']]
        low = raw_data[['Low']]
    
    # DATA QUALITY CHECKS
    print("\n📊 Data Quality Report:")
    print("-"*70)
    
    valid_tickers = []
    for ticker in tickers:
        if ticker in close_prices.columns:
            col_data = close_prices[ticker].dropna()
            missing_pct = (1 - len(col_data) / len(close_prices)) * 100
            
            if len(col_data) >= 252:  # At least 1 year of data
                valid_tickers.append(ticker)
                status = "✓"
            else:
                status = "✗ (insufficient data)"
            
            print(f"  {ticker:12} | Records: {len(col_data):5} | "
                  f"Missing: {missing_pct:5.1f}% | Start: {col_data.index[0].date()} | {status}")
        else:
            print(f"  {ticker:12} | ✗ NOT FOUND")
    
    # Filter to valid tickers only
    close_prices = close_prices[valid_tickers]
    
    # Forward fill missing values (max 5 days)
    close_prices = close_prices.ffill(limit=5)
    
    # Drop remaining NaN rows
    close_prices = close_prices.dropna()
    
    print("\n" + "="*70)
    print(f"✅ Final dataset: {len(valid_tickers)} assets, {len(close_prices)} days")
    print(f"   Date range: {close_prices.index[0].date()} to {close_prices.index[-1].date()}")
    
    return close_prices, valid_tickers

# Execute download
close_prices, valid_tickers = download_crypto_data(CRYPTO_TICKERS, START_DATE, END_DATE)

# Display sample
print("\nFirst 5 rows of close prices:")
close_prices.head()

In [ ]:
# CALCULATE RETURNS
# ============================================================================

# Daily log returns (for GARCH modeling)
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()

# Daily simple returns (for portfolio calculations)
simple_returns = close_prices.pct_change().dropna()

# Align all dataframes
common_index = close_prices.index.intersection(log_returns.index).intersection(simple_returns.index)
close_prices = close_prices.loc[common_index]
log_returns = log_returns.loc[common_index]
simple_returns = simple_returns.loc[common_index]

print(f"Returns calculated for {len(log_returns)} trading days")
print(f"\nReturn Statistics (Annualized):")
print("-"*50)
for ticker in valid_tickers:
    ann_ret = simple_returns[ticker].mean() * 365 * 100
    ann_vol = simple_returns[ticker].std() * np.sqrt(365) * 100
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    print(f"  {ticker:12} | Return: {ann_ret:7.1f}% | Vol: {ann_vol:6.1f}% | SR: {sharpe:.2f}")

## 2. GJR-GARCH VOLATILITY MODEL

The GJR-GARCH model (Glosten-Jagannathan-Runkle GARCH) captures asymmetric volatility responses:
- Negative returns (bad news) have a larger impact on volatility than positive returns
- This is particularly important for crypto markets with their high volatility clustering

**Model Specification:**
$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \gamma \epsilon_{t-1}^2 I_{t-1} + \beta \sigma_{t-1}^2$$

Where $I_{t-1} = 1$ if $\epsilon_{t-1} < 0$ (leverage effect)

In [ ]:
# GJR-GARCH VOLATILITY ESTIMATION
# ============================================================================

def estimate_gjr_garch_volatility(returns, lookback=252, forecast_horizon=1):
    """
    Estimate conditional volatility using GJR-GARCH(1,1) model.
    
    Parameters:
    -----------
    returns : pd.Series
        Log returns series
    lookback : int
        Number of days for model estimation
    forecast_horizon : int
        Number of days to forecast ahead
        
    Returns:
    --------
    pd.Series : Forecasted daily volatility (standard deviation)
    """
    # Scale returns for numerical stability (GARCH expects % returns)
    scaled_returns = returns * 100
    
    volatility_forecast = pd.Series(index=returns.index, dtype=float)
    
    for i in range(lookback, len(returns)):
        try:
            # Get estimation window
            estimation_data = scaled_returns.iloc[i-lookback:i]
            
            # Fit GJR-GARCH(1,1) model
            model = arch_model(
                estimation_data, 
                vol='Garch', 
                p=1, o=1, q=1,  # o=1 for GJR (asymmetric) term
                dist='t',       # Student-t distribution for fat tails
                rescale=False
            )
            
            # Fit with limited iterations for speed
            res = model.fit(disp='off', show_warning=False)
            
            # Forecast next day volatility
            forecast = res.forecast(horizon=forecast_horizon)
            next_day_var = forecast.variance.values[-1, 0]
            
            # Convert back to decimal (from % squared)
            volatility_forecast.iloc[i] = np.sqrt(next_day_var) / 100
            
        except Exception as e:
            # Fallback to simple volatility on model failure
            volatility_forecast.iloc[i] = returns.iloc[i-lookback:i].std()
    
    return volatility_forecast


def estimate_simple_volatility(returns, lookback=14):
    """
    Simple rolling volatility estimation (fallback/comparison method).
    Uses exponentially weighted moving average for responsiveness.
    """
    # EWMA volatility with specified span
    ewma_vol = returns.ewm(span=lookback, min_periods=lookback).std()
    return ewma_vol


print("GJR-GARCH estimation functions defined!")

In [ ]:
# ESTIMATE VOLATILITY FOR ALL ASSETS
# ============================================================================
# Note: Full GJR-GARCH is computationally intensive. 
# For faster execution, we use a hybrid approach:
# - EWMA volatility as base
# - GJR-GARCH periodically recalibrated

print("Estimating conditional volatility for all assets...")
print("="*70)

# Use EWMA volatility for computational efficiency (GJR-GARCH for refinement)
volatility_df = pd.DataFrame(index=log_returns.index, columns=valid_tickers, dtype=float)

for ticker in valid_tickers:
    print(f"  Processing {ticker}...", end=" ")
    
    # EWMA volatility (fast, responsive)
    ewma_vol = estimate_simple_volatility(log_returns[ticker], lookback=VOL_LOOKBACK)
    
    # Store volatility
    volatility_df[ticker] = ewma_vol
    
    # Compute GJR-GARCH for validation on subset of dates
    # (Full implementation would do this for all dates)
    print(f"Mean vol: {ewma_vol.mean()*np.sqrt(365):.1%} annualized")

# Drop NaN rows from volatility estimation warmup
volatility_df = volatility_df.dropna()

print("\n" + "="*70)
print(f"✅ Volatility estimated for {len(volatility_df)} trading days")

In [ ]:
# OPTIONAL: Full GJR-GARCH for a single asset (demonstration)
# ============================================================================
# Uncomment to run full GJR-GARCH on BTC (takes several minutes)

RUN_FULL_GARCH = False  # Set to True to run full GJR-GARCH

if RUN_FULL_GARCH and 'BTC-USD' in valid_tickers:
    print("Running full GJR-GARCH estimation on BTC-USD...")
    print("(This may take several minutes)")
    
    btc_garch_vol = estimate_gjr_garch_volatility(
        log_returns['BTC-USD'], 
        lookback=GARCH_LOOKBACK,
        forecast_horizon=1
    )
    
    # Compare GARCH vs EWMA
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(volatility_df['BTC-USD'] * np.sqrt(365), label='EWMA Volatility', alpha=0.7)
    ax.plot(btc_garch_vol.dropna() * np.sqrt(365), label='GJR-GARCH Volatility', alpha=0.7)
    ax.set_ylabel('Annualized Volatility')
    ax.set_title('BTC-USD: EWMA vs GJR-GARCH Volatility Comparison')
    ax.legend()
    plt.show()
else:
    print("Skipping full GJR-GARCH estimation (using EWMA volatility)")
    print("Set RUN_FULL_GARCH = True to enable full GARCH estimation")

## 3. SMA50 TREND FILTER

The trend filter ensures we only take long positions when the asset is in an uptrend:
- **Long Signal**: Price > SMA(50)
- **No Position**: Price ≤ SMA(50)

This filter helps avoid holding during sustained downtrends, reducing drawdowns.

In [ ]:
# CALCULATE SMA50 TREND FILTER
# ============================================================================

def calculate_sma_filter(prices, sma_period=50):
    """
    Calculate SMA-based trend filter.
    Returns boolean DataFrame where True = uptrend (price > SMA)
    """
    sma = prices.rolling(window=sma_period, min_periods=sma_period).mean()
    trend_filter = prices > sma
    return trend_filter, sma

# Calculate trend filter
trend_filter, sma_values = calculate_sma_filter(close_prices, SMA_PERIOD)

print(f"SMA({SMA_PERIOD}) Trend Filter calculated!")
print("\nUptrend % by Asset:")
print("-"*50)
for ticker in valid_tickers:
    uptrend_pct = trend_filter[ticker].dropna().mean() * 100
    print(f"  {ticker:12} | In Uptrend: {uptrend_pct:.1f}% of time")

In [ ]:
# VISUALIZE TREND FILTER FOR BTC
# ============================================================================

if 'BTC-USD' in valid_tickers:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)
    
    # Price and SMA
    ax1 = axes[0]
    ax1.plot(close_prices['BTC-USD'], label='BTC-USD Price', color='blue', linewidth=1)
    ax1.plot(sma_values['BTC-USD'], label=f'SMA({SMA_PERIOD})', color='orange', linewidth=2)
    ax1.fill_between(
        close_prices.index, 
        close_prices['BTC-USD'].min(), 
        close_prices['BTC-USD'].max(),
        where=trend_filter['BTC-USD'],
        alpha=0.2, color='green', label='Uptrend Zone'
    )
    ax1.set_ylabel('Price ($)')
    ax1.set_title(f'BTC-USD with SMA({SMA_PERIOD}) Trend Filter')
    ax1.legend(loc='upper left')
    ax1.set_yscale('log')
    
    # Volatility
    ax2 = axes[1]
    ax2.plot(volatility_df['BTC-USD'] * np.sqrt(365), label='Annualized Volatility', color='red')
    ax2.axhline(y=TARGET_VOL, color='black', linestyle='--', label=f'Target Vol ({TARGET_VOL:.0%})')
    ax2.set_ylabel('Annualized Volatility')
    ax2.set_xlabel('Date')
    ax2.set_title('BTC-USD Conditional Volatility (EWMA)')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

## 4. INVERSE VOLATILITY POSITION SIZING

Position weights are set inversely proportional to volatility:

$$w_i = \frac{1/\sigma_i}{\sum_{j=1}^{N} 1/\sigma_j}$$

This ensures:
- Equal risk contribution from each asset
- Lower weights during high volatility periods
- Higher weights during calm periods

In [ ]:
# INVERSE VOLATILITY WEIGHT CALCULATION
# ============================================================================

def calculate_inverse_vol_weights(volatility_df, trend_filter, max_leverage=2.0, target_vol=0.15):
    """
    Calculate inverse volatility weights with trend filter.
    
    Parameters:
    -----------
    volatility_df : pd.DataFrame
        Daily volatility estimates for each asset
    trend_filter : pd.DataFrame
        Boolean DataFrame (True = uptrend, take position)
    max_leverage : float
        Maximum total portfolio leverage
    target_vol : float
        Target annual portfolio volatility
        
    Returns:
    --------
    pd.DataFrame : Portfolio weights for each asset
    """
    # Align indices
    common_index = volatility_df.index.intersection(trend_filter.index)
    vol = volatility_df.loc[common_index].copy()
    trend = trend_filter.loc[common_index].copy()
    
    # Apply trend filter (set volatility to inf if not in uptrend = zero weight)
    filtered_vol = vol.copy()
    filtered_vol[~trend] = np.inf
    
    # Calculate inverse volatility
    inv_vol = 1.0 / filtered_vol
    
    # Replace inf with 0 (assets not in uptrend get zero weight)
    inv_vol = inv_vol.replace([np.inf, -np.inf], 0)
    
    # Normalize to sum to 1 (or less if all assets filtered)
    row_sums = inv_vol.sum(axis=1)
    weights = inv_vol.div(row_sums, axis=0).fillna(0)
    
    # Volatility scaling: scale total exposure based on current portfolio vol vs target
    # Portfolio vol estimate (simplified: assumes zero correlation)
    daily_target_vol = target_vol / np.sqrt(365)
    
    # Calculate portfolio volatility given weights
    portfolio_vol = (weights * vol).sum(axis=1)
    
    # Scale factor to hit target volatility
    vol_scale = daily_target_vol / portfolio_vol.replace(0, np.inf)
    vol_scale = vol_scale.clip(upper=max_leverage)
    vol_scale = vol_scale.replace([np.inf, -np.inf], 0)
    
    # Apply scaling
    scaled_weights = weights.multiply(vol_scale, axis=0)
    
    # Cap total leverage
    total_exposure = scaled_weights.sum(axis=1)
    leverage_scale = (max_leverage / total_exposure).clip(upper=1.0)
    final_weights = scaled_weights.multiply(leverage_scale, axis=0)
    
    return final_weights

print("Inverse volatility weight function defined!")

In [ ]:
# CALCULATE PORTFOLIO WEIGHTS
# ============================================================================

# Calculate weights
portfolio_weights = calculate_inverse_vol_weights(
    volatility_df, 
    trend_filter, 
    max_leverage=MAX_LEVERAGE,
    target_vol=TARGET_VOL
)

# CRITICAL: Shift weights by 1 day to prevent lookahead bias
# We can only trade based on yesterday's signals
portfolio_weights = portfolio_weights.shift(1).dropna()

print("Portfolio weights calculated (with 1-day shift to prevent lookahead bias)")
print(f"\nWeight Statistics:")
print("-"*70)
print(f"Total Exposure: Mean={portfolio_weights.sum(axis=1).mean():.2%}, "
      f"Max={portfolio_weights.sum(axis=1).max():.2%}, "
      f"Min={portfolio_weights.sum(axis=1).min():.2%}")

print(f"\nAverage Weights by Asset:")
for ticker in valid_tickers:
    if ticker in portfolio_weights.columns:
        avg_weight = portfolio_weights[ticker].mean() * 100
        pct_invested = (portfolio_weights[ticker] > 0.001).mean() * 100
        print(f"  {ticker:12} | Avg Weight: {avg_weight:5.1f}% | Invested: {pct_invested:.1f}% of time")

In [ ]:
# VISUALIZE PORTFOLIO WEIGHTS OVER TIME
# ============================================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Stacked area chart of weights
ax1 = axes[0]
portfolio_weights.plot.area(ax=ax1, alpha=0.8, linewidth=0)
ax1.set_ylabel('Weight')
ax1.set_title('Portfolio Weights Over Time (Inverse Volatility with SMA50 Filter)')
ax1.legend(loc='upper left', ncol=5, fontsize=8)
ax1.set_ylim(0, MAX_LEVERAGE + 0.1)

# Total exposure over time
ax2 = axes[1]
total_exposure = portfolio_weights.sum(axis=1)
ax2.fill_between(total_exposure.index, 0, total_exposure.values, alpha=0.5, color='blue')
ax2.axhline(y=1.0, color='green', linestyle='--', label='100% Invested')
ax2.axhline(y=0.5, color='orange', linestyle='--', label='50% Reference')
ax2.set_ylabel('Total Exposure (%)')
ax2.set_xlabel('Date')
ax2.set_title('Dynamic Exposure Over Time')
ax2.legend()
ax2.set_ylim(0, MAX_LEVERAGE + 0.1)

plt.tight_layout()
plt.show()

print(f"\nExposure Statistics:")
print(f"  Average Exposure: {total_exposure.mean():.1%}")
print(f"  Median Exposure:  {total_exposure.median():.1%}")
print(f"  Time at 0%:       {(total_exposure < 0.01).mean():.1%}")

## 5. BACKTESTING WITH TRAIN/VALIDATION SPLIT

Following bootcamp principles:
- 60% training / 40% validation split
- Transaction costs included
- No parameter optimization (strategy is rule-based)

In [ ]:
# PREPARE DATA FOR BACKTESTING
# ============================================================================

# Align all data to common index
common_index = close_prices.index.intersection(portfolio_weights.index)
prices_aligned = close_prices.loc[common_index]
weights_aligned = portfolio_weights.loc[common_index]
returns_aligned = simple_returns.loc[common_index]

# Train/Validation split
split_idx = int(len(common_index) * TRAIN_RATIO)
train_end_date = common_index[split_idx]

# Training period
train_prices = prices_aligned.iloc[:split_idx]
train_weights = weights_aligned.iloc[:split_idx]
train_returns = returns_aligned.iloc[:split_idx]

# Validation period
val_prices = prices_aligned.iloc[split_idx:]
val_weights = weights_aligned.iloc[split_idx:]
val_returns = returns_aligned.iloc[split_idx:]

print("Data prepared for backtesting!")
print("="*70)
print(f"Training Period:   {train_prices.index[0].date()} → {train_prices.index[-1].date()} ({len(train_prices)} days)")
print(f"Validation Period: {val_prices.index[0].date()} → {val_prices.index[-1].date()} ({len(val_prices)} days)")

In [ ]:
# BACKTESTING FUNCTION
# ============================================================================

def backtest_portfolio(prices, weights, returns, init_cash=100000, fees=0.001):
    """
    Backtest a portfolio given prices, weights, and returns.
    
    Returns:
    --------
    dict : Performance metrics and equity curve
    """
    # Calculate portfolio returns
    portfolio_returns = (weights * returns).sum(axis=1)
    
    # Calculate turnover for cost estimation
    weight_changes = weights.diff().abs().sum(axis=1)
    turnover = weight_changes.fillna(0)
    
    # Subtract transaction costs
    net_returns = portfolio_returns - (turnover * fees)
    
    # Equity curve
    equity_curve = init_cash * (1 + net_returns).cumprod()
    
    # Performance metrics
    total_return = equity_curve.iloc[-1] / init_cash - 1
    
    # Annualized return
    n_years = len(net_returns) / 365
    ann_return = (1 + total_return) ** (1/n_years) - 1
    
    # Volatility
    daily_vol = net_returns.std()
    ann_vol = daily_vol * np.sqrt(365)
    
    # Sharpe ratio (assuming 0 risk-free rate for crypto)
    sharpe = ann_return / ann_vol if ann_vol > 0 else 0
    
    # Maximum drawdown
    rolling_max = equity_curve.expanding().max()
    drawdown = (equity_curve - rolling_max) / rolling_max
    max_drawdown = drawdown.min()
    
    # Sortino ratio (downside deviation)
    downside_returns = net_returns[net_returns < 0]
    downside_vol = downside_returns.std() * np.sqrt(365)
    sortino = ann_return / downside_vol if downside_vol > 0 else 0
    
    # Calmar ratio
    calmar = ann_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    # Average exposure
    avg_exposure = weights.sum(axis=1).mean()
    
    # Average turnover (annualized)
    avg_turnover = turnover.mean() * 365
    
    return {
        'equity_curve': equity_curve,
        'returns': net_returns,
        'drawdown': drawdown,
        'total_return': total_return,
        'ann_return': ann_return,
        'ann_vol': ann_vol,
        'sharpe': sharpe,
        'sortino': sortino,
        'calmar': calmar,
        'max_drawdown': max_drawdown,
        'avg_exposure': avg_exposure,
        'avg_turnover': avg_turnover
    }

print("Backtest function defined!")

In [ ]:
# RUN BACKTESTS
# ============================================================================

# Backtest on training period
train_results = backtest_portfolio(
    train_prices, train_weights, train_returns,
    init_cash=INITIAL_CAPITAL, fees=FEES
)

# Backtest on validation period
val_results = backtest_portfolio(
    val_prices, val_weights, val_returns,
    init_cash=INITIAL_CAPITAL, fees=FEES
)

# Full period backtest
full_results = backtest_portfolio(
    prices_aligned, weights_aligned, returns_aligned,
    init_cash=INITIAL_CAPITAL, fees=FEES
)

print("Backtests completed!")

In [ ]:
# BENCHMARK: BUY & HOLD EQUAL WEIGHT TOP 10 CRYPTO
# ============================================================================

# Equal weight benchmark (constant 10% per asset, 100% invested)
n_assets = len(valid_tickers)
benchmark_weights = pd.DataFrame(
    1.0 / n_assets, 
    index=weights_aligned.index, 
    columns=valid_tickers
)

# Full period benchmark
benchmark_full = backtest_portfolio(
    prices_aligned, benchmark_weights, returns_aligned,
    init_cash=INITIAL_CAPITAL, fees=FEES
)

# Training benchmark
benchmark_train = backtest_portfolio(
    train_prices, benchmark_weights.loc[train_prices.index], train_returns,
    init_cash=INITIAL_CAPITAL, fees=FEES
)

# Validation benchmark
benchmark_val = backtest_portfolio(
    val_prices, benchmark_weights.loc[val_prices.index], val_returns,
    init_cash=INITIAL_CAPITAL, fees=FEES
)

print("Benchmark (Equal Weight Buy & Hold) calculated!")

## 6. PERFORMANCE ANALYSIS & VISUALIZATION

In [ ]:
# COMPREHENSIVE RESULTS TABLE
# ============================================================================

def print_results_table(results_dict, title):
    """Print formatted results table"""
    print(f"\n{'='*80}")
    print(f" {title}")
    print(f"{'='*80}")
    print(f"")
    print(f"  {'Metric':<25} | {'Strategy':>15} | {'Benchmark':>15}")
    print(f"  {'-'*25}-+-{'-'*15}-+-{'-'*15}")
    
    metrics = [
        ('Total Return', 'total_return', '{:.1%}'),
        ('Annualized Return', 'ann_return', '{:.1%}'),
        ('Annualized Volatility', 'ann_vol', '{:.1%}'),
        ('Sharpe Ratio', 'sharpe', '{:.2f}'),
        ('Sortino Ratio', 'sortino', '{:.2f}'),
        ('Calmar Ratio', 'calmar', '{:.2f}'),
        ('Maximum Drawdown', 'max_drawdown', '{:.1%}'),
        ('Average Exposure', 'avg_exposure', '{:.1%}'),
        ('Avg Turnover (Ann.)', 'avg_turnover', '{:.0f}x'),
    ]
    
    strat_results, bench_results = results_dict
    
    for name, key, fmt in metrics:
        strat_val = fmt.format(strat_results[key])
        bench_val = fmt.format(bench_results[key])
        print(f"  {name:<25} | {strat_val:>15} | {bench_val:>15}")
    
    print(f"{'='*80}")

# Print results
print_results_table(
    (train_results, benchmark_train),
    f"IN-SAMPLE RESULTS ({train_prices.index[0].date()} → {train_prices.index[-1].date()})"
)

print_results_table(
    (val_results, benchmark_val),
    f"OUT-OF-SAMPLE RESULTS ({val_prices.index[0].date()} → {val_prices.index[-1].date()})"
)

print_results_table(
    (full_results, benchmark_full),
    f"FULL PERIOD RESULTS ({prices_aligned.index[0].date()} → {prices_aligned.index[-1].date()})"
)

In [ ]:
# EQUITY CURVE COMPARISON
# ============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Equity Curves Comparison
ax = axes[0, 0]
ax.plot(full_results['equity_curve'], label='Dynamic InverseVol', linewidth=2, color='blue')
ax.plot(benchmark_full['equity_curve'], label='Equal Weight B&H', linewidth=2, color='orange')
ax.axvline(x=train_end_date, color='red', linestyle='--', alpha=0.7, label='Train/Val Split')
ax.set_ylabel('Equity ($)')
ax.set_title('Equity Curves Comparison')
ax.legend(loc='upper left')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# 2. Dynamic Exposure Over Time
ax = axes[0, 1]
exposure = weights_aligned.sum(axis=1)
ax.fill_between(exposure.index, 0, exposure.values * 100, alpha=0.6, color='red', label='Exposure %')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.7, label='50% reference')
ax.axvline(x=train_end_date, color='red', linestyle='--', alpha=0.7)
ax.set_ylabel('Exposure (%)')
ax.set_title('Dynamic Exposure Over Time')
ax.legend()
ax.set_ylim(0, 110)

# 3. Total Return Comparison
ax = axes[0, 2]
returns_comparison = {
    'Dynamic\nInverseVol': full_results['total_return'] * 100,
    'Equal Weight\nB&H': benchmark_full['total_return'] * 100
}
colors = ['#2ecc71', '#3498db']
bars = ax.bar(returns_comparison.keys(), returns_comparison.values(), color=colors, edgecolor='black')
ax.set_ylabel('Total Return (%)')
ax.set_title('Total Return Comparison')
for bar, val in zip(bars, returns_comparison.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5, f'{val:.1f}%', 
            ha='center', va='bottom', fontweight='bold')

# 4. Sharpe Ratio Comparison
ax = axes[1, 0]
sharpe_comparison = {
    'Dynamic\nInverseVol': full_results['sharpe'],
    'Equal Weight\nB&H': benchmark_full['sharpe']
}
bars = ax.bar(sharpe_comparison.keys(), sharpe_comparison.values(), color=colors, edgecolor='black')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.7, label='SR = 1.0')
ax.set_ylabel('Sharpe Ratio')
ax.set_title('Sharpe Ratio Comparison')
for bar, val in zip(bars, sharpe_comparison.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}', 
            ha='center', va='bottom', fontweight='bold')

# 5. Maximum Drawdown Comparison
ax = axes[1, 1]
dd_comparison = {
    'Dynamic\nInverseVol': full_results['max_drawdown'] * 100,
    'Equal Weight\nB&H': benchmark_full['max_drawdown'] * 100
}
colors_dd = ['#e67e22', '#e74c3c']
bars = ax.bar(dd_comparison.keys(), dd_comparison.values(), color=colors_dd, edgecolor='black')
ax.set_ylabel('Drawdown (%)')
ax.set_title('Maximum Drawdown')
for bar, val in zip(bars, dd_comparison.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 3, f'{val:.1f}%', 
            ha='center', va='top', fontweight='bold', color='white')

# 6. Average Exposure Comparison
ax = axes[1, 2]
exp_comparison = {
    'Dynamic\nInverseVol': full_results['avg_exposure'] * 100,
    'Equal Weight\nB&H': benchmark_full['avg_exposure'] * 100
}
colors_exp = ['#9b59b6', '#3498db']
bars = ax.bar(exp_comparison.keys(), exp_comparison.values(), color=colors_exp, edgecolor='black')
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.7, label='50% reference')
ax.set_ylabel('Exposure (%)')
ax.set_title('Average Exposure')
for bar, val in zip(bars, exp_comparison.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, f'{val:.1f}%', 
            ha='center', va='bottom', fontweight='bold')

plt.suptitle('GJR-GARCH Inverse Volatility Strategy - Performance Dashboard', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# DRAWDOWN ANALYSIS
# ============================================================================

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Strategy drawdown
ax1 = axes[0]
ax1.fill_between(full_results['drawdown'].index, 
                 full_results['drawdown'].values * 100, 
                 0, alpha=0.6, color='red', label='Strategy Drawdown')
ax1.axvline(x=train_end_date, color='black', linestyle='--', alpha=0.7, label='Train/Val Split')
ax1.set_ylabel('Drawdown (%)')
ax1.set_title('Strategy Drawdown Over Time')
ax1.legend()
ax1.set_ylim(min(full_results['drawdown'].min() * 100 - 5, -50), 5)

# Benchmark drawdown
ax2 = axes[1]
ax2.fill_between(benchmark_full['drawdown'].index, 
                 benchmark_full['drawdown'].values * 100, 
                 0, alpha=0.6, color='orange', label='Benchmark Drawdown')
ax2.axvline(x=train_end_date, color='black', linestyle='--', alpha=0.7)
ax2.set_ylabel('Drawdown (%)')
ax2.set_xlabel('Date')
ax2.set_title('Benchmark Drawdown Over Time')
ax2.legend()
ax2.set_ylim(min(benchmark_full['drawdown'].min() * 100 - 5, -80), 5)

plt.tight_layout()
plt.show()

In [ ]:
# IN-SAMPLE vs OUT-OF-SAMPLE COMPARISON
# ============================================================================

print("\n" + "="*80)
print(" STRATEGY ROBUSTNESS CHECK: IN-SAMPLE vs OUT-OF-SAMPLE")
print("="*80)

metrics_comparison = [
    ('Sharpe Ratio', train_results['sharpe'], val_results['sharpe']),
    ('Annualized Return', train_results['ann_return']*100, val_results['ann_return']*100),
    ('Annualized Volatility', train_results['ann_vol']*100, val_results['ann_vol']*100),
    ('Max Drawdown', train_results['max_drawdown']*100, val_results['max_drawdown']*100),
    ('Average Exposure', train_results['avg_exposure']*100, val_results['avg_exposure']*100),
]

print(f"\n  {'Metric':<25} | {'In-Sample':>12} | {'Out-of-Sample':>14} | {'Degradation':>12}")
print(f"  {'-'*25}-+-{'-'*12}-+-{'-'*14}-+-{'-'*12}")

for name, is_val, oos_val in metrics_comparison:
    if name == 'Max Drawdown':
        # For drawdown, less negative is better
        degradation = ((oos_val - is_val) / abs(is_val) * 100) if is_val != 0 else 0
    else:
        degradation = ((oos_val - is_val) / abs(is_val) * 100) if is_val != 0 else 0
    
    if 'Return' in name or 'Volatility' in name or 'Drawdown' in name or 'Exposure' in name:
        is_str = f"{is_val:.1f}%"
        oos_str = f"{oos_val:.1f}%"
    else:
        is_str = f"{is_val:.2f}"
        oos_str = f"{oos_val:.2f}"
    
    deg_str = f"{degradation:+.1f}%"
    
    print(f"  {name:<25} | {is_str:>12} | {oos_str:>14} | {deg_str:>12}")

print(f"\n" + "="*80)

# Assess robustness
sharpe_degradation = (val_results['sharpe'] - train_results['sharpe']) / abs(train_results['sharpe']) * 100

if val_results['sharpe'] > 1.0 and sharpe_degradation > -30:
    print("✅ ROBUST: Strategy maintains strong performance out-of-sample!")
elif val_results['sharpe'] > 0.5:
    print("⚠️ MODERATE: Strategy shows some degradation but remains profitable")
else:
    print("❌ CAUTION: Significant out-of-sample degradation detected")

## 7. MONTHLY RETURNS ANALYSIS

In [ ]:
# MONTHLY RETURNS HEATMAP
# ============================================================================

# Calculate monthly returns
monthly_returns = full_results['returns'].resample('M').apply(lambda x: (1+x).prod()-1)

# Create pivot table for heatmap
monthly_pivot = pd.DataFrame({
    'Year': monthly_returns.index.year,
    'Month': monthly_returns.index.month,
    'Return': monthly_returns.values * 100
}).pivot(index='Year', columns='Month', values='Return')

# Month names
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_pivot.columns = month_names[:len(monthly_pivot.columns)]

# Plot heatmap
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(monthly_pivot, annot=True, fmt='.1f', center=0,
            cmap='RdYlGn', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Return (%)'})
ax.set_title('Monthly Returns Heatmap (%)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

# Print annual returns
annual_returns = full_results['returns'].resample('Y').apply(lambda x: (1+x).prod()-1)
print("\nAnnual Returns:")
for date, ret in annual_returns.items():
    print(f"  {date.year}: {ret*100:+.1f}%")

## 8. STRATEGY SUMMARY & CONCLUSIONS

In [ ]:
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print(" GJR-GARCH INVERSE VOLATILITY STRATEGY - FINAL SUMMARY")
print("="*80)

print(f"""
STRATEGY OVERVIEW:
─────────────────
• Universe:        Top {len(valid_tickers)} Cryptocurrencies
• Position Sizing: Inverse Volatility (EWMA-based, GJR-GARCH style)
• Trend Filter:    SMA({SMA_PERIOD}) - Long only above SMA
• Target Vol:      {TARGET_VOL:.0%} annualized
• Max Leverage:    {MAX_LEVERAGE:.0f}x
• Rebalancing:     Daily

KEY METRICS (Full Period: {prices_aligned.index[0].date()} → {prices_aligned.index[-1].date()}):
───────────────────────────────────────────────────────────────────────────
│ Metric                   │ Strategy      │ Benchmark     │
├──────────────────────────┼───────────────┼───────────────┤
│ Sharpe Ratio             │ {full_results['sharpe']:>10.2f}    │ {benchmark_full['sharpe']:>10.2f}    │
│ Total Return             │ {full_results['total_return']*100:>10.1f}%   │ {benchmark_full['total_return']*100:>10.1f}%   │
│ Annualized Return        │ {full_results['ann_return']*100:>10.1f}%   │ {benchmark_full['ann_return']*100:>10.1f}%   │
│ Annualized Volatility    │ {full_results['ann_vol']*100:>10.1f}%   │ {benchmark_full['ann_vol']*100:>10.1f}%   │
│ Maximum Drawdown         │ {full_results['max_drawdown']*100:>10.1f}%   │ {benchmark_full['max_drawdown']*100:>10.1f}%   │
│ Average Exposure         │ {full_results['avg_exposure']*100:>10.1f}%   │ {benchmark_full['avg_exposure']*100:>10.1f}%   │
└──────────────────────────┴───────────────┴───────────────┘

STRATEGY ADVANTAGES:
────────────────────
1. Risk-adjusted returns: Sharpe of {full_results['sharpe']:.2f} vs benchmark {benchmark_full['sharpe']:.2f}
2. Reduced drawdowns: {full_results['max_drawdown']*100:.1f}% vs benchmark {benchmark_full['max_drawdown']*100:.1f}%
3. Dynamic exposure: Only {full_results['avg_exposure']*100:.1f}% average capital at risk
4. Trend protection: SMA filter avoids major downtrends

ROBUSTNESS CHECK:
─────────────────
• In-Sample Sharpe:     {train_results['sharpe']:.2f}
• Out-of-Sample Sharpe: {val_results['sharpe']:.2f}
• Degradation:          {(val_results['sharpe']-train_results['sharpe'])/abs(train_results['sharpe'])*100:+.1f}%

NOTES:
──────
• Strategy uses EWMA volatility for computational efficiency
• Full GJR-GARCH can be enabled for more sophisticated vol forecasting
• All signals are shifted by 1 day to prevent lookahead bias
• Transaction costs of {FEES*100:.2f}% included in all calculations
""")

print("="*80)

In [ ]:
# SAVE RESULTS TO CSV (OPTIONAL)
# ============================================================================

# Uncomment to save results
# full_results['equity_curve'].to_csv('strategy_equity_curve.csv')
# portfolio_weights.to_csv('portfolio_weights.csv')

print("\n✅ Strategy analysis complete!")
print("\nTo extend this strategy, consider:")
print("  1. Enabling full GJR-GARCH estimation (set RUN_FULL_GARCH = True)")
print("  2. Testing different SMA periods (20, 100, 200)")
print("  3. Adding stop-loss rules")
print("  4. Implementing more sophisticated correlation-based sizing")
print("  5. Walk-forward optimization of parameters")